# Compare Annotation-Only Variants to Manual Meta-Analysis Results

This notebook compares automated meta-analysis results from three runs against manual NeuroMetaBench meta-analyses:

- `annotation-only`
- `annotation-only-2`
- `annotation-only-lc`

It computes two similarity metrics between automated and manual maps:

- Dice coefficient on thresholded images (`z > 1.96`)
- Pearson correlation coefficient (`r`) on full unthresholded images


In [ ]:
import os
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from pathlib import Path
from nilearn import plotting
import json

## Setup Paths

In [ ]:
repos_dir = '/home/zorro/repos'

# Automated meta-analysis runs to compare
automated_runs = [
    'annotation-only',
    'annotation-only-2',
    'annotation-only-lc',
]

base_folder = 'autonima-results/projects/social/coordinates'
auto_runs = {
    run_name: {
        'meta_results': os.path.join(base_folder, run_name, 'outputs', 'meta_analysis_results'),
        'annotation_json': os.path.join(base_folder, run_name, 'outputs', 'nimads_annotation.json'),
    }
    for run_name in automated_runs
}

# Manual meta-analysis paths (NeuroMetaBench)
social_meta_path = 'autonima-results/projects/social/coordinates/analysis/nmb-results/social'
social_nimads_path = 'neurometabench/data/nimads/social'

# Mapping from NeuroMetaBench social analysis names to autonima-results social analysis names
mappings = {
    'affiliation_merged': 'affiliation_attachment',
    'self_merged': 'perception_self',
    'others_merged': 'perception_others',
    'soccomm_merged': 'social_communication',
    'all_merged': 'social_processing_all',
}

# Result file to load
result_file = 'z.nii.gz'

# Threshold for Dice coefficient (z-score threshold)
dice_threshold = 1.96

# Original single-run annotation path retained for count-summary cells below
annotation_only_annotation_path = auto_runs['annotation-only']['annotation_json']

Path('images').mkdir(exist_ok=True)


## Load Manual Meta-Analysis Data

In [ ]:
def load_nifti_data(base_dir, subdir, filename):
    """Load a NIfTI file and return the data array."""
    filepath = os.path.join(base_dir, subdir, filename)
    if not os.path.exists(filepath):
        print(f"Warning: File not found: {filepath}")
        return None, None
    
    img = nib.load(filepath)
    data = img.get_fdata()
    return data, filepath


# Load manual meta-analysis results
manual_data = {}
manual_paths = {}
for manual_name in mappings.keys():
    manual_dir = os.path.join(repos_dir, social_meta_path, manual_name)
    data, filepath = load_nifti_data(repos_dir, os.path.join(social_meta_path, manual_name), result_file)
    if data is not None:
        manual_data[manual_name] = data
        manual_paths[manual_name] = filepath

print(f"\nLoaded {len(manual_data)} manual meta-analysis maps")



## Load Automated NIfTI Files (3 Runs)

In [ ]:
# Load automated results for each run
automated_data = {}
automated_paths = {}

for run_name, run_paths in auto_runs.items():
    run_data = {}
    run_path_map = {}
    run_dir = os.path.join(repos_dir, run_paths['meta_results'])

    print(f"\n[{run_name}]")
    if not os.path.exists(run_dir):
        print(f"  Warning: directory not found: {run_dir}")
        automated_data[run_name] = run_data
        automated_paths[run_name] = run_path_map
        continue

    available_annotations = sorted([
        d for d in os.listdir(run_dir)
        if os.path.isdir(os.path.join(run_dir, d))
    ])
    print(f"  Found {len(available_annotations)} annotation types")
    print(f"  {available_annotations}")

    for annotation_name in available_annotations:
        data, filepath = load_nifti_data(
            repos_dir,
            os.path.join(run_paths['meta_results'], annotation_name),
            result_file
        )
        if data is not None:
            run_data[annotation_name] = data
            run_path_map[annotation_name] = filepath

    print(f"  Loaded {len(run_data)} maps")
    automated_data[run_name] = run_data
    automated_paths[run_name] = run_path_map


## Create Common Mask

In [ ]:
def create_common_mask(automated_data_by_run, data_dict_manual):
    """Create a mask of voxels that are valid (finite) across all loaded images."""
    masks = []

    for _, data_dict in automated_data_by_run.items():
        for _, data in data_dict.items():
            masks.append(np.isfinite(data))

    for _, data in data_dict_manual.items():
        masks.append(np.isfinite(data))

    if len(masks) == 0:
        return None

    common_mask = masks[0]
    for mask in masks[1:]:
        common_mask = common_mask & mask

    print(f"Common mask: {np.sum(common_mask)} valid voxels")
    return common_mask

# Create common mask (including all automated runs + manual data)
common_mask = create_common_mask(automated_data, manual_data)
if common_mask is None:
    raise ValueError('No images were loaded; cannot create a common mask.')

# Apply mask to all data
automated_data_masked = {
    run_name: {name: data.flatten()[common_mask.flatten()] for name, data in data_dict.items()}
    for run_name, data_dict in automated_data.items()
}
manual_data_masked = {
    name: data.flatten()[common_mask.flatten()]
    for name, data in manual_data.items()
}


## Compute Similarity Metrics (Dice and Pearson r)

In [ ]:
def compute_dice_coefficient(img1, img2, threshold=0):
    """Compute Dice coefficient between two thresholded binary masks."""
    binary1 = img1 > threshold
    binary2 = img2 > threshold

    intersection = np.sum(binary1 & binary2)
    sum_volumes = np.sum(binary1) + np.sum(binary2)

    if sum_volumes == 0:
        return 0.0

    return (2.0 * intersection) / sum_volumes

def compute_pearson_r(img1, img2):
    """Compute Pearson r on unthresholded vectors."""
    finite = np.isfinite(img1) & np.isfinite(img2)
    if not np.any(finite):
        return np.nan

    x = img1[finite]
    y = img2[finite]

    if x.size < 2 or y.size < 2:
        return np.nan

    if np.all(x == x[0]) or np.all(y == y[0]):
        return np.nan

    r, _ = pearsonr(x, y)
    return r

def compute_similarity_matrix(data_dict_1, data_dict_2, metric='dice', threshold=0):
    """Compute pairwise similarity matrix between two sets of vectors."""
    names_1 = list(data_dict_1.keys())
    names_2 = list(data_dict_2.keys())

    mat = np.full((len(names_1), len(names_2)), np.nan)
    for i, name_1 in enumerate(names_1):
        for j, name_2 in enumerate(names_2):
            vec_1 = data_dict_1[name_1]
            vec_2 = data_dict_2[name_2]
            if metric == 'dice':
                val = compute_dice_coefficient(vec_1, vec_2, threshold=threshold)
            elif metric == 'r':
                val = compute_pearson_r(vec_1, vec_2)
            else:
                raise ValueError(f"Unknown metric: {metric}")
            mat[i, j] = val

    return mat, names_1, names_2


## Compare Automated Runs to Manual Meta-Analyses

In [ ]:
# Align manual maps to automated annotation names (values of `mappings`)
manual_aligned = {}
for manual_name, auto_name in mappings.items():
    if manual_name in manual_data_masked:
        manual_aligned[auto_name] = manual_data_masked[manual_name]

# Align each automated run to the same key space
aligned_auto_by_run = {}
for run_name, data_run in automated_data_masked.items():
    aligned = {}
    for manual_name, auto_name in mappings.items():
        if auto_name in data_run:
            aligned[auto_name] = data_run[auto_name]
    aligned_auto_by_run[run_name] = aligned

print('Aligned maps:')
print(f"  Manual: {len(manual_aligned)} maps")
for run_name in automated_runs:
    print(f"  {run_name}: {len(aligned_auto_by_run.get(run_name, {}))} maps")

# Pairwise auto-vs-manual matrices for each run
dice_auto_vs_manual = {}
r_auto_vs_manual = {}
names_auto_vs_manual = {}

for run_name in automated_runs:
    aligned_auto = aligned_auto_by_run.get(run_name, {})

    dice_mat, row_names, col_names = compute_similarity_matrix(
        aligned_auto, manual_aligned, metric='dice', threshold=dice_threshold
    )
    r_mat, _, _ = compute_similarity_matrix(
        aligned_auto, manual_aligned, metric='r'
    )

    dice_auto_vs_manual[run_name] = dice_mat
    r_auto_vs_manual[run_name] = r_mat
    names_auto_vs_manual[run_name] = (row_names, col_names)

print('\nMatrix shapes (auto vs manual):')
for run_name in automated_runs:
    print(f"  {run_name}: dice={dice_auto_vs_manual[run_name].shape}, r={r_auto_vs_manual[run_name].shape}")

# Diagonal (matched annotation) metrics across runs
annotation_order = [auto_name for _, auto_name in mappings.items() if auto_name in manual_aligned]
diag_rows = []

for annotation_name in annotation_order:
    manual_vec = manual_aligned.get(annotation_name)
    for run_name in automated_runs:
        auto_vec = aligned_auto_by_run.get(run_name, {}).get(annotation_name)

        dice_val = np.nan
        r_val = np.nan
        if auto_vec is not None and manual_vec is not None:
            dice_val = compute_dice_coefficient(auto_vec, manual_vec, threshold=dice_threshold)
            r_val = compute_pearson_r(auto_vec, manual_vec)

        diag_rows.append({
            'annotation': annotation_name,
            'run': run_name,
            'dice': dice_val,
            'r': r_val,
            'available': auto_vec is not None and manual_vec is not None,
        })

diag_metrics_df = pd.DataFrame(diag_rows)
diag_metrics_df['annotation'] = pd.Categorical(diag_metrics_df['annotation'], categories=annotation_order, ordered=True)
diag_metrics_df['run'] = pd.Categorical(diag_metrics_df['run'], categories=automated_runs, ordered=True)
diag_metrics_df = diag_metrics_df.sort_values(['annotation', 'run']).reset_index(drop=True)

print('\nDiagonal metrics (matched automated annotation vs corresponding manual map):')
display(diag_metrics_df)

print('\nDice (pivot):')
display(diag_metrics_df.pivot(index='annotation', columns='run', values='dice').round(3))

print('\nPearson r (pivot):')
display(diag_metrics_df.pivot(index='annotation', columns='run', values='r').round(3))

missing_by_run = {
    run_name: [a for a in annotation_order if a not in aligned_auto_by_run.get(run_name, {})]
    for run_name in automated_runs
}
print('\nMissing mapped annotations by run:')
for run_name, missing in missing_by_run.items():
    print(f"  {run_name}: {missing if missing else 'None'}")


## Grouped Bar Plots: Matched Manual vs Automated Similarity

In [ ]:
def plot_grouped_metric(df, metric_col, title, ylabel, ylim=None, center_zero=False, filename=None):
    plot_df = df.copy()
    plot_df['annotation_label'] = plot_df['annotation'].astype(str)

    fig, ax = plt.subplots(figsize=(13, 6))
    sns.barplot(
        data=plot_df,
        x='annotation_label',
        y=metric_col,
        hue='run',
        hue_order=automated_runs,
        dodge=True,
        estimator='mean',
        errorbar=None,
        ax=ax,
    )

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Annotation (matched automated ↔ manual)', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

    if ylim is not None:
        ax.set_ylim(*ylim)
    if center_zero:
        ax.axhline(0, color='black', linewidth=1, alpha=0.5)

    ax.legend(title='Automated run', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()

    if filename:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

plot_grouped_metric(
    diag_metrics_df,
    metric_col='dice',
    title=f'Dice (z > {dice_threshold}) for Matched Manual vs Automated Maps',
    ylabel='Dice coefficient',
    ylim=(0, 1),
    filename='images/dice_grouped_manual_vs_automated_3runs.png',
)

plot_grouped_metric(
    diag_metrics_df,
    metric_col='r',
    title='Pearson r (Unthresholded) for Matched Manual vs Automated Maps',
    ylabel='Pearson r',
    ylim=(-1, 1),
    center_zero=True,
    filename='images/pearson_r_grouped_manual_vs_automated_3runs.png',
)


## Summary Statistics for Matched Manual vs Automated Comparisons

In [ ]:
summary_by_run = (
    diag_metrics_df.groupby('run', observed=False)[['dice', 'r']]
    .agg(['mean', 'median', 'std', 'count'])
    .round(3)
)

display(summary_by_run)


## Compare studies and analyses included

In [ ]:
import os
import json
def compute_annotation_counts(annotation_path):
    """Load annotation JSON and compute counts per annotation type."""
    with open(os.path.join(repos_dir, annotation_path), 'r') as f:
        meta = json.load(f)

    # Handle list format
    if isinstance(meta, list):
        meta = meta[0]

    notes = meta["notes"]

    # Count analyses per boolean column
    analyses = {k: set() for k in meta["note_keys"].keys()}
    pmids = {k: set() for k in meta["note_keys"].keys()}
    for note in notes:
        analysis = note['analysis']
        pmid, _, analysis_id = analysis.split("_")
        for k in analyses:
            try:
                if bool(note['note'].get(k, False)):
                    analyses[k].add(analysis)
                    pmids[k].add(pmid)
            except Exception:
                pass

    pmids = {k: {p for p in v if p is not None} for k, v in pmids.items()}
    return analyses, pmids, list(analyses.keys())


In [ ]:
analyses, pmids, annotation_names = compute_annotation_counts(annotation_only_annotation_path)
pmids_per_annotation = {k: len(v) for k, v in pmids.items()}
analyses_per_annotation = {k: len(v) for k, v in analyses.items()}

In [ ]:
pmids_per_annotation

In [ ]:
analyses_per_annotation

### Load manual meta-analysis counts (NeuroMetaBench)

## Full Dice and R Matrices (Auto vs Manual, by Run)

In [ ]:
# Show full pairwise matrices (diagonal + off-diagonal) for Dice and Pearson r for each run

full_dice_matrices = {}
full_r_matrices = {}

print('Full Similarity Matrices (Automated vs Manual)')
print('=' * 70)

for run_name in automated_runs:
    aligned_auto = aligned_auto_by_run.get(run_name, {})
    print(f'\nRun: {run_name}')

    auto_names = [auto_name for _, auto_name in mappings.items() if auto_name in aligned_auto]
    manual_names = [manual_name for manual_name in mappings.keys() if manual_name in manual_data_masked]

    if not auto_names:
        print('  No mapped automated annotations found for this run. Skipping matrices.')
        full_dice_matrices[run_name] = pd.DataFrame(columns=manual_names)
        full_r_matrices[run_name] = pd.DataFrame(columns=manual_names)
        continue

    dice_matrix_df = pd.DataFrame(index=auto_names, columns=manual_names, dtype=float)
    r_matrix_df = pd.DataFrame(index=auto_names, columns=manual_names, dtype=float)

    for auto_name in auto_names:
        for manual_name in manual_names:
            auto_vec = aligned_auto.get(auto_name)
            manual_vec = manual_data_masked.get(manual_name)
            if auto_vec is None or manual_vec is None:
                dice_matrix_df.loc[auto_name, manual_name] = np.nan
                r_matrix_df.loc[auto_name, manual_name] = np.nan
                continue

            dice_matrix_df.loc[auto_name, manual_name] = compute_dice_coefficient(
                auto_vec, manual_vec, threshold=dice_threshold
            )
            r_matrix_df.loc[auto_name, manual_name] = compute_pearson_r(auto_vec, manual_vec)

    full_dice_matrices[run_name] = dice_matrix_df
    full_r_matrices[run_name] = r_matrix_df

    print('\nDice matrix (thresholded at z > {:.2f}):'.format(dice_threshold))
    display(dice_matrix_df.round(3))

    print('Pearson r matrix (unthresholded):')
    display(r_matrix_df.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    sns.heatmap(
        dice_matrix_df.astype(float),
        annot=True, fmt='.3f', cmap='RdYlGn',
        vmin=0, vmax=1, ax=axes[0], cbar_kws={'label': 'Dice'}
    )
    axes[0].set_title(f'Dice Matrix\n{run_name} (Auto) vs Manual', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Manual Meta-Analysis', fontweight='bold')
    axes[0].set_ylabel('Automated Annotation', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].tick_params(axis='y', rotation=0)

    sns.heatmap(
        r_matrix_df.astype(float),
        annot=True, fmt='.3f', cmap='coolwarm', center=0,
        vmin=-1, vmax=1, ax=axes[1], cbar_kws={'label': 'Pearson r'}
    )
    axes[1].set_title(f'Pearson r Matrix\n{run_name} (Auto) vs Manual', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Manual Meta-Analysis', fontweight='bold')
    axes[1].set_ylabel('Automated Annotation', fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].tick_params(axis='y', rotation=0)

    plt.tight_layout()
    plt.savefig(f'images/similarity_matrices_auto_vs_manual_{run_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
